# 🔬 Model Pipeline: Google Gemma-2-2B-IT (4-bit QLoRA)
### **Jadavpur University Research Internship | Under: Prof. Tohida Rehman**
**Track:** FIRE MixSub-SciHigh Scientific Highlight Summarization
**Model ID:** `google/gemma-2-2b-it`
**Architecture:** CAUSAL (4-bit QLoRA)

---
### 🎯 Strict Uniform Benchmark Constraints (Scientifically Fair)
- **`MAX_SOURCE_LENGTH = 512` tokens**: Uniform input length across all 6 models.
- **`MAX_TARGET_LENGTH = 128` tokens**: Uniform output generation limit across all 6 models.

### 📊 The 3 Non-Redundant Benchmark Pillars Measured:
1. **Overall Quality**: ROUGE-L F1 (↑ Higher is Better) & Target F1 Score (↑)
2. **Hallucination Risk**: Ungrounded Token % (↓ Lower is Better) & Precision Source (% grounded in abstract)
3. **Inference Time**: Wall-Clock Latency per Sample (↓ Lower is Faster) & Throughput (tok/s ↑)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json

BASE_DIR = "/content/drive/MyDrive/JU_Benchmark_Project"
DATASET_DIR = os.path.join(BASE_DIR, "Dataset")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODEL_DIR = os.path.join(BASE_DIR, "checkpoints", "gemma-2-2b")

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128

print("Drive ready!")

In [ ]:
!pip -q install transformers accelerate bitsandbytes peft sentencepiece rouge-score

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU: Runtime > Change runtime type > T4 GPU")

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

In [ ]:

# CELL 3 — SELF-CONTAINED DATASET LOADER

import os
import pandas as pd
import gdown

# Define folders again
BASE_DIR = "/content/drive/MyDrive/JU_Benchmark_Project"
DATASET_DIR = os.path.join(BASE_DIR, "Dataset")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

TRAIN_CSV = os.path.join(DATASET_DIR, "MixSub-SciHigh_train_FIRE.csv")
VAL_CSV = os.path.join(DATASET_DIR, "MixSub-SciHigh_val_FIRE.csv")
TEST_CSV = os.path.join(DATASET_DIR, "MixSub-SciHigh_test_FIRE(Masked).csv")

# Download missing datasets
downloads = {
    TRAIN_CSV: "1oydpEnDZ8Chu7dQpT5RGnnRTMvhfq-8Q",
    VAL_CSV: "1-A0tc0sFJGgwWEOywM9Ddyxi6G4K_Xhj"
}

for path, file_id in downloads.items():
    if not os.path.exists(path):
        print("Downloading:", os.path.basename(path))
        gdown.download(
            f"https://drive.google.com/uc?id={file_id}",
            path,
            quiet=False
        )

# Verify validation dataset
if not os.path.exists(VAL_CSV):
    raise FileNotFoundError("Validation CSV download failed.")

val_df = pd.read_csv(VAL_CSV)
val_df = val_df.dropna(
    subset=["Abstract", "Highlights"]
).reset_index(drop=True)

# Optional masked test dataset
test_df = None

if os.path.exists(TEST_CSV):
    test_df = pd.read_csv(TEST_CSV)
    test_df["Abstract"] = test_df["Abstract"].fillna("").astype(str)
    print("Masked test dataset found!")
else:
    print("Masked test dataset not found. Skipping test predictions.")

print("\nSUCCESS!")
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df) if test_df is not None else 0)
print("Results folder:", RESULTS_DIR)


In [ ]:
from getpass import getpass
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

HF_TOKEN = getpass("Paste your Hugging Face access token: ")
login(token=HF_TOKEN)

MODEL_ID = "google/gemma-2-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading Gemma 2 2B...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    token=HF_TOKEN,
    low_cpu_mem_usage=True
)

model.eval()

print("Gemma loaded successfully!")
print("Model device:", model.device)

In [ ]:

# ============================================================
# CELL 5: GEMMA 2 2B — FAST BENCHMARK + PREDICTIONS
# ============================================================

import os
import re
import json
import time
import torch
import numpy as np
import pandas as pd

from collections import Counter
from rouge_score import rouge_scorer

# ------------------------------------------------------------
# 1. INITIALIZE ALL SETTINGS
# ------------------------------------------------------------

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128

MAX_EVAL_SAMPLES = 5
MAX_TEST_SAMPLES = 5

MODEL_ID = globals().get("MODEL_ID", "google/gemma-2-2b-it")

RESULTS_DIR = globals().get(
    "RESULTS_DIR",
    "/content/drive/MyDrive/JU_Benchmark_Project/results"
)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Verify required objects
if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError("Run Cell 4 first to load Gemma.")

if "val_df" not in globals():
    raise RuntimeError("Run Cell 3 first to load validation data.")

if val_df.empty:
    raise ValueError("Validation dataset is empty.")

if "Abstract" not in val_df.columns or "Highlights" not in val_df.columns:
    raise ValueError("Validation CSV needs Abstract and Highlights columns.")

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Gemma benchmark initialized!")
print("Model:", MODEL_ID)
print("Validation samples:", min(MAX_EVAL_SAMPLES, len(val_df)))


# ------------------------------------------------------------
# 2. PROMPT GENERATION
# ------------------------------------------------------------

def make_prompt(abstract):

    messages = [
        {
            "role": "user",
            "content": (
                "Generate concise, accurate scientific research "
                "highlights based only on the following abstract. "
                "Do not add unsupported information.\n\n"
                + str(abstract)
            )
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


# Preserve the final assistant-generation prompt when truncating.
tokenizer.truncation_side = "left"


# ------------------------------------------------------------
# 3. GENERATION FUNCTION
# ------------------------------------------------------------

def generate_one(abstract):

    prompt = make_prompt(abstract)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=MAX_SOURCE_LENGTH,
        truncation=True
    ).to(model.device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            do_sample=False,
            num_beams=1,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    generated_tokens = output[
        0, inputs["input_ids"].shape[1]:
    ]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return prediction, elapsed, len(generated_tokens)


# ------------------------------------------------------------
# 4. VALIDATION BENCHMARK
# ------------------------------------------------------------

eval_df = val_df.head(MAX_EVAL_SAMPLES).copy()

predictions = []
latencies = []
token_counts = []

print("\nStarting Gemma validation benchmark...\n")

for i, abstract in enumerate(eval_df["Abstract"]):

    pred, latency, tokens = generate_one(abstract)

    predictions.append(pred)
    latencies.append(latency)
    token_counts.append(tokens)

    print(
        f"Sample {i+1}/{len(eval_df)} | "
        f"Latency: {latency:.2f}s | "
        f"Tokens: {tokens}"
    )


# ------------------------------------------------------------
# 5. QUALITY METRICS
# ------------------------------------------------------------

scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

rouge_scores = [
    scorer.score(
        str(ref),
        str(pred)
    )["rougeL"].fmeasure

    for ref, pred in zip(
        eval_df["Highlights"],
        predictions
    )
]


def tokenize_words(text):

    return re.findall(
        r"\b[a-zA-Z0-9]+\b",
        str(text).lower()
    )


def overlap_metrics(pred, source, reference):

    p = Counter(tokenize_words(pred))
    s = Counter(tokenize_words(source))
    r = Counter(tokenize_words(reference))

    total_p = sum(p.values())
    total_r = sum(r.values())

    source_overlap = sum((p & s).values())
    target_overlap = sum((p & r).values())

    source_precision = (
        source_overlap / total_p if total_p else 0
    )

    target_precision = (
        target_overlap / total_p if total_p else 0
    )

    target_recall = (
        target_overlap / total_r if total_r else 0
    )

    f1 = (
        2 * target_precision * target_recall /
        (target_precision + target_recall)
        if target_precision + target_recall else 0
    )

    return source_precision, f1


scores = [
    overlap_metrics(p, s, r)

    for p, s, r in zip(
        predictions,
        eval_df["Abstract"],
        eval_df["Highlights"]
    )
]

avg_grounding = float(
    np.mean([x[0] for x in scores])
)

avg_f1 = float(
    np.mean([x[1] for x in scores])
)


# ------------------------------------------------------------
# 6. CREATE BENCHMARK REPORT
# ------------------------------------------------------------

benchmark = {

    "model": "gemma-2-2b",
    "model_id": MODEL_ID,

    "evaluation_samples": len(eval_df),

    "overall_quality_rougeL_f1": float(
        np.mean(rouge_scores)
    ),

    "f1_score_target": avg_f1,

    "source_grounding_precision": avg_grounding,

    # Lexical proxy, NOT a verified hallucination rate.
    "lexical_non_grounded_fraction": 1.0 - avg_grounding,

    "wall_clock_latency_sec": float(
        np.mean(latencies)
    ),

    "total_wall_clock_sec": float(
        np.sum(latencies)
    ),

    "throughput_tokens_per_sec": float(
        sum(token_counts) / max(sum(latencies), 1e-6)
    ),

    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,

    "fine_tuning": False,
    "quantization": "4-bit NF4",
    "evaluation_type": "Zero-shot inference"
}


# ------------------------------------------------------------
# 7. SAVE BENCHMARK JSON
# ------------------------------------------------------------

benchmark_path = os.path.join(
    RESULTS_DIR,
    "benchmark_gemma-2-2b.json"
)

with open(benchmark_path, "w") as f:

    json.dump(
        benchmark,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 8. SAVE VALIDATION PREDICTIONS
# ------------------------------------------------------------

val_output = eval_df.copy()

val_output["Predicted_Highlights"] = predictions

val_path = os.path.join(
    RESULTS_DIR,
    "gemma_validation_predictions.csv"
)

val_output.to_csv(
    val_path,
    index=False
)


# ------------------------------------------------------------
# 9. PRINT BENCHMARK
# ------------------------------------------------------------

print("\n========================================")
print("       GEMMA 2 2B BENCHMARK RESULTS")
print("========================================")

for key, value in benchmark.items():

    print(f"{key}: {value}")

print("\nValidation results saved successfully!")


# ------------------------------------------------------------
# 10. MASKED TEST PREDICTIONS
# ------------------------------------------------------------

if "test_df" in globals() and test_df is not None:

    if len(test_df) > 0 and "Abstract" in test_df.columns:

        test_run = test_df.head(MAX_TEST_SAMPLES).copy()

        test_predictions = []

        print("\nGenerating masked test predictions...\n")

        for i, abstract in enumerate(test_run["Abstract"]):

            pred, _, _ = generate_one(abstract)

            test_predictions.append(pred)

            print(
                f"Test sample {i+1}/{len(test_run)} completed"
            )

        test_run["Highlights"] = test_predictions

        test_path = os.path.join(
            RESULTS_DIR,
            "MixSub-SciHigh_test_predictions_gemma-2-2b.csv"
        )

        test_run.to_csv(
            test_path,
            index=False
        )

        print("\nTest predictions saved:", test_path)

    else:
        print("\nTest dataset is empty. Skipping.")

else:
    print("\nMasked test dataset unavailable. Skipping test predictions.")


# ------------------------------------------------------------
# 11. FINAL STATUS
# ------------------------------------------------------------

print("\n========================================")
print("          EXECUTION COMPLETED")
print("========================================")

print("Benchmark JSON:", benchmark_path)
print("Validation CSV:", val_path)
print("Results directory:", RESULTS_DIR)

print("\nAvailable output files:")

for filename in os.listdir(RESULTS_DIR):
    if "gemma" in filename.lower():
        print(" -", filename)

print("\nSUCCESS!")
